In [40]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [41]:
pip install wandb

Note: you may need to restart the kernel to use updated packages.


In [42]:
from kaggle_secrets import UserSecretsClient
import wandb

user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("WANDB_API_KEY")

wandb.login(key=api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [43]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
import wandb
import os

In [44]:
MAX_LEN = 256
BATCH_SIZE = 16
EPOCHS = 15
LEARNING_RATE = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
vocab_size = tokenizer.vocab_size

Using device: cpu


In [45]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
sample_submission_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')


In [46]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_test = is_test
        self.options = ['A', 'B', 'C', 'D', 'E']
        self.label_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        input_ids_list = []
        
        for opt in self.options:
            option_text = str(row[opt])
            combined_text = prompt + " [SEP] " + option_text
            
            encoded = self.tokenizer(combined_text,padding='max_length', truncation=True, max_length=self.max_len, return_tensors='pt')
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            
        input_ids_tensor = torch.stack(input_ids_list)
        item = {'input_ids': input_ids_tensor}
        
        if not self.is_test:
            item['label'] = torch.tensor(self.label_map[row['answer']], dtype=torch.long)
            
        return item

In [47]:
class CustomTextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, num_filters=100, filter_sizes=[3, 4, 5], num_classes=1):
        super(CustomTextCNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, num_classes)

    def forward(self, input_ids):
        batch_size, num_choices, seq_len = input_ids.shape
        x = input_ids.view(-1, seq_len)
        embedded = self.embedding(x).transpose(1, 2) 
        
        conv_results = []
        for conv in self.convs:
            c = torch.relu(conv(embedded))
            c = torch.max_pool1d(c, c.size(2)).squeeze(2)
            conv_results.append(c)
            
        out = torch.cat(conv_results, dim=1) 
        out = self.dropout(out)
        
        logits = self.fc(out)
        
        reshaped_logits = logits.view(batch_size, num_choices)
        return reshaped_logits

In [48]:
def mapk(actual, predicted, k=3):
    score = 0.0
    for a, p in zip(actual, predicted):
        p = p[:k]
        if a in p:
            score += 1.0 / (p.index(a) + 1)
    return score / len(actual)

In [49]:
def train_model():
    val_size = int(0.2 * len(train_df))
    train_data = train_df.iloc[:-val_size]
    val_data = train_df.iloc[-val_size:]
    
    train_dataset = MCQDataset(train_data, tokenizer, MAX_LEN)
    val_dataset = MCQDataset(val_data, tokenizer, MAX_LEN)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
  
    model = CustomTextCNN(vocab_size=vocab_size).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    
    wandb.init(
        project="23f3002112-t22026", 
        name="Custom-CNN-Scratch",
        config={"architecture": "CNN", "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE}
    )
    wandb.watch(model, criterion, log="all", log_freq=10)
    
    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        correct_preds = 0
        total_preds = 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            
            optimizer.zero_grad()
            logits = model(input_ids)
            
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            correct_preds += (preds == labels).sum().item()
            total_preds += labels.size(0)
            
        train_acc = correct_preds / total_preds
        train_loss = total_loss / len(train_loader)
        
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        actual_answers=[]
        predicted_answers=[]
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(DEVICE)
                labels = batch['label'].to(DEVICE)
                
                logits = model(input_ids)
                loss = criterion(logits, labels)
                
                val_loss += loss.item()
                preds = torch.argmax(logits, dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                top3 = torch.topk(logits, k=3,dim=1).indices.cpu().tolist()
                actual_answers.extend(labels.cpu().tolist())
                predicted_answers.extend(top3)
                
        val_acc = val_correct / val_total
        val_map3 = mapk(actual_answers,predicted_answers,k=3)
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | MAP@3:{val_map3:.4f}")
        
        # Log to WandB
        wandb.log({
            "Train Loss": train_loss, 
            "Train Accuracy": train_acc, 
            "Val Loss": avg_val_loss, 
            "Val Accuracy": val_acc,
            "Val MAP@3":val_map3,
            "Epoch": epoch + 1
        })
        
    torch.save(model.state_dict(), "cnn_mcq_model.pt")
    wandb.save("cnn_mcq_model.pt")
    
    return model


In [50]:
def generate_submission(model):
    print("Generating submission.csv...")
    test_dataset = MCQDataset(test_df, tokenizer, MAX_LEN, is_test=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
    
    model.eval()
    all_predictions = []
    
    options_map = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(DEVICE)
            logits = model(input_ids)
            
            top_3_indices = torch.topk(logits, k=3, dim=1).indices.cpu().numpy()
            
            for indices in top_3_indices:
                labels = [options_map[idx] for idx in indices]
                all_predictions.append(" ".join(labels))
 
    submission_df = pd.DataFrame({
        'id': test_df['id'],
        'Prediction': all_predictions
    })
    
    submission_df.to_csv('submission.csv', index=False)
    print("submission.csv successfully saved!")


In [51]:
trained_model = train_model()
generate_submission(trained_model)
wandb.finish()

Epoch 1/15 | Train Loss: 1.3738 | Train Acc: 0.4331 | Val Loss: 0.8182 | Val Acc: 0.9475 | MAP@3:0.9696
Epoch 2/15 | Train Loss: 0.7893 | Train Acc: 0.7094 | Val Loss: 0.4367 | Val Acc: 0.9850 | MAP@3:0.9900
Epoch 3/15 | Train Loss: 0.5153 | Train Acc: 0.8244 | Val Loss: 0.2507 | Val Acc: 0.9950 | MAP@3:0.9967
Epoch 4/15 | Train Loss: 0.3988 | Train Acc: 0.8706 | Val Loss: 0.1623 | Val Acc: 0.9925 | MAP@3:0.9954
Epoch 5/15 | Train Loss: 0.2629 | Train Acc: 0.9106 | Val Loss: 0.1059 | Val Acc: 0.9925 | MAP@3:0.9962
Epoch 6/15 | Train Loss: 0.2198 | Train Acc: 0.9256 | Val Loss: 0.0798 | Val Acc: 0.9950 | MAP@3:0.9975
Epoch 7/15 | Train Loss: 0.1781 | Train Acc: 0.9475 | Val Loss: 0.0574 | Val Acc: 1.0000 | MAP@3:1.0000
Epoch 8/15 | Train Loss: 0.1510 | Train Acc: 0.9537 | Val Loss: 0.0433 | Val Acc: 0.9975 | MAP@3:0.9988
Epoch 9/15 | Train Loss: 0.1011 | Train Acc: 0.9669 | Val Loss: 0.0314 | Val Acc: 1.0000 | MAP@3:1.0000
Epoch 10/15 | Train Loss: 0.0899 | Train Acc: 0.9781 | Val Loss:

wandb: WARNING Symlinked 1 file into the W&B run directory; call wandb.save again to sync new files.


Epoch 15/15 | Train Loss: 0.0453 | Train Acc: 0.9831 | Val Loss: 0.0061 | Val Acc: 1.0000 | MAP@3:1.0000
Generating submission.csv...
submission.csv successfully saved!


Epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
Train Accuracy,▁▅▆▇▇▇█████████
Train Loss,█▅▃▃▂▂▂▂▁▁▁▁▁▁▁
Val Accuracy,▁▆▇▇▇▇█████████
Val Loss,█▅▃▂▂▂▁▁▁▁▁▁▁▁▁
Val MAP@3,▁▆▇▇▇▇█████████
Epoch,15
Train Accuracy,0.98313
Train Loss,0.04527
Val Accuracy,1
Val Loss,0.00609
